# 05 · Compare & Ensemble
Load all three trained models, produce the comparison figures, run TTA, build the confidence-weighted ensemble, evaluate on the held-out TEST set **exactly once**, and export the summary table + all report figures.

In [ ]:
# === Preamble 1/2: environment & GPU report ===
# This is a REMOTE Colab kernel — it cannot see your local files.
import sys
print('Python:', sys.version.split()[0])
try:
    import torch
    print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('bfloat16 supported:', torch.cuda.is_bf16_supported())
    else:
        print('No GPU — Runtime > Change runtime type > A100 (or L4).')
except ImportError:
    print('torch installs in the next cell.')

In [ ]:
# === Preamble 2/2: clone-or-pull + install (+ optional autoreload) ===
import os, subprocess, sys

REPO_URL = "https://github.com/Kidhurshan/plant-leaf-classifier.git"  # <-- EDIT to your repo
REPO_DIR = "/content/plant-leaf-classifier"
# Private repo? use https://<TOKEN>@github.com/Kidhurshan/plant-leaf-classifier.git

if not os.path.isdir(REPO_DIR):
    print('Cloning', REPO_URL)
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                'requirements.txt'], check=True)

# Hot-reload src/ after a `git pull` (optional convenience; never fatal).
try:
    from IPython import get_ipython
    _ip = get_ipython()
    _ip.run_line_magic('load_ext', 'autoreload')
    _ip.run_line_magic('autoreload', '2')
    print('autoreload enabled.')
except Exception as _e:
    print('autoreload not enabled (non-fatal):', repr(_e))

from src.utils import sync_repo, gpu_report
sync_repo()   # git pull + print the commit hash these results are traceable to
gpu_report()

## Config & datasets

In [ ]:
from src.config import load_config
from src.utils import set_seed, detect_amp, count_parameters
from src.data import prepare_datasets
from src.augment import GPUAugment
from src import viz
import torch
from pathlib import Path

SMOKE = False
cfg = load_config('configs/default.yaml')
cfg.paths.ensure_dirs(); set_seed(cfg.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
amp = detect_amp(device)
datasets, class_names = prepare_datasets(cfg, device, smoke=SMOKE)
aug_eval = GPUAugment(cfg.augment, cfg.data.img_size, device, training=False)
bs = cfg.smoke.batch_size if SMOKE else cfg.train.batch_size
suffix = '_smoke' if SMOKE else ''

## Combined training curves (visual 5)

In [ ]:
from src.engine import read_history_csv
histories = {k: read_history_csv(f'{cfg.paths.metrics_dir}/{k}{suffix}_history.csv')
             for k in cfg.model_list}
if all(histories.values()):
    viz.plot_combined_curves(histories,
        out_path=f'{cfg.paths.figures_dir}/combined_curves.png');

## Evaluate each model on TEST (once), with & without TTA
Confusion matrix (visual 6) per model; collects per-class F1 and ensemble inputs.

In [ ]:
from src.models import load_trained_model
from src.evaluate import (compute_metrics, collect_predictions,
                          save_confusion_csv, save_classification_report)
from src.tta import compare_tta
from src.engine import read_history_csv

per_model, test_probs, val_f1, targets = {}, {}, {}, None
for key in cfg.model_list:
    ck = Path(cfg.paths.checkpoint_dir)/f'{key}{suffix}_best.pt'
    model = load_trained_model(cfg, key, ck, device)
    vp = collect_predictions(model, datasets['val'], aug_eval, amp, bs)
    val_f1[key] = compute_metrics(vp['targets'], vp['preds'], class_names)['macro_f1']
    tta = compare_tta(model, datasets['test'], aug_eval, amp, bs, class_names)
    targets = tta['targets']; test_probs[key] = tta['without']['probs']
    tm = compute_metrics(targets, test_probs[key].argmax(1), class_names)
    viz.plot_confusion_matrix(tm['confusion_matrix'], class_names,
        out_path=f'{cfg.paths.figures_dir}/{key}_confusion.png',
        title=f'Confusion — {viz.display_name(key)}')
    save_confusion_csv(tm['confusion_matrix'], class_names,
        f'{cfg.paths.metrics_dir}/{key}_confusion.csv')
    save_classification_report(tm['report'],
        f'{cfg.paths.metrics_dir}/{key}_test_report.txt')
    hist = read_history_csv(f'{cfg.paths.metrics_dir}/{key}{suffix}_history.csv')
    per_model[key] = dict(accuracy=tm['accuracy'],
        macro_precision=tm['macro_precision'], macro_recall=tm['macro_recall'],
        macro_f1=tm['macro_f1'], acc_tta=tta['with']['accuracy'],
        f1_tta=tta['with']['macro_f1'], params=count_parameters(model)['total'],
        train_time_s=sum(h['time_s'] for h in hist) if hist else float('nan'),
        per_class_f1=tm['per_class_f1'])
    print(f"{key}: test acc={tm['accuracy']:.4f} macroF1={tm['macro_f1']:.4f}")
    del model
    if device.type == 'cuda': torch.cuda.empty_cache()

## Per-class F1 (visual 7)

In [ ]:
viz.plot_per_class_f1({k: per_model[k]['per_class_f1'] for k in cfg.model_list},
    class_names, out_path=f'{cfg.paths.figures_dir}/per_class_f1.png');

## TTA gain (visual 9)

In [ ]:
viz.plot_tta_gain({k: {'without': per_model[k]['macro_f1'],
                       'with': per_model[k]['f1_tta']} for k in cfg.model_list},
    out_path=f'{cfg.paths.figures_dir}/tta_gain.png');

## Confidence-weighted ensemble (visual 6 — ensemble CM)

In [ ]:
from src.ensemble import build_ensemble
ens = build_ensemble(test_probs, val_f1, targets, class_names)
print('ensemble weights (∝ val macro-F1):', ens['weights'])
viz.plot_confusion_matrix(ens['confusion_matrix'], class_names,
    out_path=f'{cfg.paths.figures_dir}/ensemble_confusion.png',
    title='Confusion — Ensemble')
save_confusion_csv(ens['confusion_matrix'], class_names,
    f'{cfg.paths.metrics_dir}/ensemble_confusion.csv')
per_model['ensemble'] = dict(accuracy=ens['accuracy'],
    macro_precision=ens['macro_precision'], macro_recall=ens['macro_recall'],
    macro_f1=ens['macro_f1'], acc_tta=float('nan'), f1_tta=float('nan'),
    params=sum(per_model[k]['params'] for k in cfg.model_list),
    train_time_s=sum(per_model[k]['train_time_s'] for k in cfg.model_list),
    per_class_f1=ens['per_class_f1'])

## Summary table (visual 8) — the report centrepiece

In [ ]:
from src.report import write_summary_table
paths = write_summary_table(per_model, cfg.model_list + ['ensemble'],
                            cfg.paths.metrics_dir, cfg.paths.figures_dir)
print(open(paths['md']).read())

## Grad-CAM grid (visual 10)
Correct vs incorrect examples per class for the proposed model.

In [ ]:
from src.gradcam import generate_gradcam_examples
proposed = cfg.model_list[-1]
gm = load_trained_model(cfg, proposed,
    Path(cfg.paths.checkpoint_dir)/f'{proposed}{suffix}_best.pt', device)
items = generate_gradcam_examples(gm, datasets['test'], class_names,
    aug_eval, cfg.data.img_size, cfg.paths.gradcam_dir)
viz.plot_gradcam_grid(items,
    out_path=f'{cfg.paths.figures_dir}/gradcam_grid.png');

## t-SNE of penultimate features (visual 11)

In [ ]:
feat = collect_predictions(gm, datasets['test'], aug_eval, amp, bs,
                           return_features=True)
viz.plot_tsne(feat['features'], feat['targets'], class_names,
    out_path=f'{cfg.paths.figures_dir}/tsne_features.png',
    title=f't-SNE — {viz.display_name(proposed)} features', seed=cfg.seed);

## Outputs

In [ ]:
print('All figures ->', cfg.paths.figures_dir)
print('Summary     ->', cfg.paths.metrics_dir + '/summary_table.md')
print('Grad-CAM    ->', cfg.paths.gradcam_dir)

---
### ⚠️ When finished: disconnect and DELETE the runtime
`Runtime > Disconnect and delete runtime`. Colab compute units are consumed the whole time a runtime is connected.